In [13]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [14]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"       
TAG      = "tio_cnnpre"                                 


COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_roi_{TAG}"
RESULTS    = f"D:/mamba_model/v7_roi_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 4
NUM_WORKERS = 0          

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_roi_tio_cnnpre
  roi_mri: 560 files
  roi_pet: 560 files


In [15]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [17]:
import sys, importlib
MEDICALNET_ROOT    = 'F:/mamba_model/MedicalNet'
MEDICALNET_WEIGHTS = 'D:/mamba_model/pretrained/resnet_10_23dataset.pth'
if MEDICALNET_ROOT not in sys.path:
    sys.path.insert(0, MEDICALNET_ROOT)
importlib.invalidate_caches()


def _load_medicalnet_state():
    try:
        ckpt = torch.load(MEDICALNET_WEIGHTS, map_location='cpu', weights_only=True)
    except Exception:
        ckpt = torch.load(MEDICALNET_WEIGHTS, map_location='cpu')
    return {k.replace('module.', ''): v for k, v in ckpt['state_dict'].items()}


class MedicalNetEncoder(nn.Module):
    """Pretrained MedicalNet ResNet-10."""
    def __init__(self, n_rois=6, d_model=32, freeze=False):
        super().__init__()
        from models import resnet
        net = resnet.resnet10(sample_input_D=64, sample_input_H=64, sample_input_W=64,
                              num_seg_classes=2, no_cuda=(device.type != 'cuda'))
        net.load_state_dict(_load_medicalnet_state(), strict=False)

        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1, self.layer2 = net.layer1, net.layer2
        self.layer3, self.layer4 = net.layer3, net.layer4
        if freeze:
            for p in self.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(512, d_model)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        with torch.no_grad():
            self.roi_embed.weight.mul_(0.02)

    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.proj(self.pool(x).flatten(1))
        return x.reshape(B, n, -1) + self.roi_embed.weight[None]


class CNNMambaBranch(nn.Module):
    def __init__(self, encoder_cls, n_rois=6, d_model=32, n_layers=2, **kw):
        super().__init__()
        self.cnn = encoder_cls(n_rois=n_rois, d_model=d_model, **kw)
        self.vim = VimEncoder(d_model, n_layers)
    def forward(self, rois):
        return self.vim(self.cnn(rois)).mean(dim=1)


def make_cnn_models(encoder_cls, **enc_kw):
    class _Uni(nn.Module):
        def __init__(self, n_rois=6, d_model=32, n_layers=2,
                     n_classes=2, d_state=16, dropout=0.4, **_):
            super().__init__()
            self.branch = CNNMambaBranch(encoder_cls, n_rois, d_model, n_layers, **enc_kw)
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(d_model, n_classes)
        def forward(self, rois):
            return self.classifier(self.dropout(self.branch(rois)))

    class _MM(nn.Module):
        def __init__(self, n_rois=6, d_model=32, n_layers=2,
                     n_classes=2, d_state=16, dropout=0.4, **_):
            super().__init__()
            self.mri_branch = CNNMambaBranch(encoder_cls, n_rois, d_model, n_layers, **enc_kw)
            self.pet_branch = CNNMambaBranch(encoder_cls, n_rois, d_model, n_layers, **enc_kw)
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(d_model * 2, n_classes)
        def forward(self, mri, pet):
            f = torch.cat([self.mri_branch(mri), self.pet_branch(pet)], dim=1)
            return self.classifier(self.dropout(f))
    return _Uni, _MM


VisionMambaModel, MultimodalVisionMambaModel = make_cnn_models(
    MedicalNetEncoder, freeze=False)

# Built on CPU so does not compete with anything on the GPU
_m = VisionMambaModel()
_sd = _load_medicalnet_state()
_pre = (torch.allclose(_m.branch.cnn.stem[0].weight.detach(), _sd['conv1.weight'])
        and torch.allclose(_m.branch.cnn.layer4[0].conv1.weight.detach(),
                           _sd['layer4.0.conv1.weight']))
with torch.no_grad():
    _ = _m(torch.randn(1, 6, 1, 64, 64, 64))
print(f"MedicalNet + Mamba, 6 tokens | params {sum(p.numel() for p in _m.parameters()):,}")
print(f"pretrained trunk loaded and trainable: {_pre}")
del _m, _sd, _pre

MedicalNet + Mamba, 6 tokens | params 14,399,714
pretrained trunk loaded and trainable: True


In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a


class ROIDataset(Dataset):
    """Single modality. Training set includes 3 augmented copies per subject."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = load_cached(f"{self.cache_dir}/{key}_{ver}.npy")
        return torch.from_numpy(a).unsqueeze(1), torch.tensor(lab, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    """Pairs MRI and PET for the same subject and the same augmentation seed."""
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mri_loaders = (dl(ROIDataset(X_train, y_train, MRI_CACHE, True, True), True),
               dl(ROIDataset(X_val,   y_val,   MRI_CACHE, True, False)),
               dl(ROIDataset(X_test,  y_test,  MRI_CACHE, True, False)))

pet_loaders = (dl(ROIDataset(X_train, y_train, PET_CACHE, False, True), True),
               dl(ROIDataset(X_val,   y_val,   PET_CACHE, False, False)),
               dl(ROIDataset(X_test,  y_test,  PET_CACHE, False, False)))

mm_loaders  = (dl(MultimodalROIDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
               dl(MultimodalROIDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
               dl(MultimodalROIDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mri_loaders[0].dataset)}")

t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
cold = time.time() - t0

t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
warm = time.time() - t0

print(f"20 batches: cold {cold:.1f}s -> warm {warm:.1f}s")
print(f"cached arrays: {len(_CACHE)}  (~{sum(a.nbytes for a in _CACHE.values())/1e9:.1f} GB)")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 batches: cold 4.2s -> warm 3.6s
cached arrays: 156  (~1.0 GB)


In [6]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)

def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot/len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))

def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)

def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may not have trained")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m*1000, "flops": fl}

results = {"mri": [], "pet": [], "mm": []}

In [7]:
results["mri"].append(run_seed(1, VisionMambaModel, mri_loaders, False, "v7_roi_cnnpre_mri"))

  MedicalNet: 12 missing, 0 unexpected keys


F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us


--- v7_roi_cnnpre_mri seed 1 ---
  ep   1 | train 0.7383 | val 0.6766 | acc 0.575 tpr 0.250 tnr 0.900 | 31s
  ep   5 | train 0.5496 | val 0.6788 | acc 0.600 tpr 0.600 tnr 0.600 | 13s
  ep  10 | train 0.4936 | val 0.6902 | acc 0.650 tpr 0.600 tnr 0.700 | 12s
  ep  15 | train 0.3561 | val 0.7419 | acc 0.700 tpr 0.550 tnr 0.850 | 12s
  ep  20 | train 0.2732 | val 0.7312 | acc 0.700 tpr 0.500 tnr 0.900 | 12s
  ep  25 | train 0.2406 | val 0.8442 | acc 0.700 tpr 0.450 tnr 0.950 | 12s
  early stop 25, best 3
  >>> TEST Acc=67.5% TPR=40.0% TNR=95.0% | params=14,399,714 train=5.5min inf=9.04ms 106.37GFLOPs best_ep=3


In [8]:
results["mri"].append(run_seed(7, VisionMambaModel, mri_loaders, False, "v7_roi_cnnpre_mri"))

F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_mri seed 7 ---
  ep   1 | train 0.7057 | val 0.6788 | acc 0.600 tpr 0.700 tnr 0.500 | 13s
  ep   5 | train 0.5701 | val 0.6714 | acc 0.675 tpr 0.650 tnr 0.700 | 12s
  ep  10 | train 0.4822 | val 0.7631 | acc 0.600 tpr 0.300 tnr 0.900 | 12s
  ep  15 | train 0.3939 | val 0.8334 | acc 0.625 tpr 0.400 tnr 0.850 | 13s
  ep  20 | train 0.2837 | val 0.8156 | acc 0.600 tpr 0.400 tnr 0.800 | 13s
  ep  25 | train 0.2209 | val 0.8790 | acc 0.600 tpr 0.400 tnr 0.800 | 13s
  early stop 27, best 12
  >>> TEST Acc=57.5% TPR=80.0% TNR=35.0% | params=14,399,714 train=5.6min inf=7.50ms 106.37GFLOPs best_ep=12


In [9]:
results["mri"].append(run_seed(123, VisionMambaModel, mri_loaders, False, "v7_roi_cnnpre_mri"))

  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_mri seed 123 ---


F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  ep   1 | train 0.7460 | val 0.7030 | acc 0.500 tpr 1.000 tnr 0.000 | 13s
  ep   5 | train 0.5426 | val 0.6177 | acc 0.650 tpr 0.850 tnr 0.450 | 14s
  ep  10 | train 0.4378 | val 0.6024 | acc 0.675 tpr 0.550 tnr 0.800 | 12s
  ep  15 | train 0.3172 | val 0.7144 | acc 0.675 tpr 0.450 tnr 0.900 | 12s
  ep  20 | train 0.2701 | val 0.7628 | acc 0.650 tpr 0.350 tnr 0.950 | 12s
  ep  25 | train 0.2273 | val 0.7871 | acc 0.675 tpr 0.550 tnr 0.800 | 12s
  early stop 25, best 3
  >>> TEST Acc=75.0% TPR=85.0% TNR=65.0% | params=14,399,714 train=5.2min inf=6.86ms 106.37GFLOPs best_ep=3


In [10]:
results["pet"].append(run_seed(1, VisionMambaModel, pet_loaders, False, "v7_roi_cnnpre_pet"))

  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_pet seed 1 ---


F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  ep   1 | train 0.7214 | val 0.6414 | acc 0.650 tpr 0.400 tnr 0.900 | 38s
  ep   5 | train 0.5501 | val 0.7036 | acc 0.700 tpr 0.450 tnr 0.950 | 12s
  ep  10 | train 0.4431 | val 0.6114 | acc 0.725 tpr 0.800 tnr 0.650 | 13s
  ep  15 | train 0.3345 | val 0.7812 | acc 0.700 tpr 0.700 tnr 0.700 | 13s
  ep  20 | train 0.2972 | val 0.6727 | acc 0.700 tpr 0.700 tnr 0.700 | 13s
  ep  25 | train 0.2503 | val 0.6119 | acc 0.775 tpr 0.750 tnr 0.800 | 12s
  ep  30 | train 0.1711 | val 0.6621 | acc 0.725 tpr 0.850 tnr 0.600 | 13s
  early stop 33, best 18
  >>> TEST Acc=62.5% TPR=55.0% TNR=70.0% | params=14,399,714 train=7.3min inf=9.37ms 106.37GFLOPs best_ep=18


In [11]:
results["pet"].append(run_seed(7, VisionMambaModel, pet_loaders, False, "v7_roi_cnnpre_pet"))

F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_pet seed 7 ---
  ep   1 | train 0.6748 | val 0.6203 | acc 0.700 tpr 0.700 tnr 0.700 | 13s
  ep   5 | train 0.5417 | val 0.5741 | acc 0.775 tpr 0.650 tnr 0.900 | 12s
  ep  10 | train 0.4441 | val 0.6948 | acc 0.725 tpr 0.550 tnr 0.900 | 13s
  ep  15 | train 0.3563 | val 0.6586 | acc 0.775 tpr 0.550 tnr 1.000 | 12s
  ep  20 | train 0.2539 | val 0.6392 | acc 0.725 tpr 0.700 tnr 0.750 | 13s
  ep  25 | train 0.2454 | val 0.7916 | acc 0.650 tpr 0.850 tnr 0.450 | 12s
  early stop 25, best 4
  >>> TEST Acc=70.0% TPR=55.0% TNR=85.0% | params=14,399,714 train=5.2min inf=6.86ms 106.37GFLOPs best_ep=4


In [12]:
results["pet"].append(run_seed(123, VisionMambaModel, pet_loaders, False, "v7_roi_cnnpre_pet"))

  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_pet seed 123 ---


F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  ep   1 | train 0.7164 | val 0.6180 | acc 0.650 tpr 0.650 tnr 0.650 | 12s
  ep   5 | train 0.5550 | val 0.6364 | acc 0.725 tpr 0.950 tnr 0.500 | 12s
  ep  10 | train 0.4200 | val 0.6550 | acc 0.700 tpr 0.850 tnr 0.550 | 13s
  ep  15 | train 0.3924 | val 0.5929 | acc 0.725 tpr 0.850 tnr 0.600 | 12s
  ep  20 | train 0.2526 | val 0.6821 | acc 0.725 tpr 0.800 tnr 0.650 | 13s
  ep  25 | train 0.1831 | val 0.6804 | acc 0.750 tpr 0.750 tnr 0.750 | 13s
  early stop 25, best 7
  >>> TEST Acc=75.0% TPR=70.0% TNR=80.0% | params=14,399,714 train=5.3min inf=7.31ms 106.37GFLOPs best_ep=7


In [13]:
results["mm"].append(run_seed(1, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_cnnpre_mm"))

  MedicalNet: 12 missing, 0 unexpected keys


F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_mm seed 1 ---
  ep   1 | train 0.6759 | val 0.6525 | acc 0.650 tpr 0.650 tnr 0.650 | 28s
  ep   5 | train 0.5338 | val 0.7662 | acc 0.575 tpr 0.850 tnr 0.300 | 25s
  ep  10 | train 0.4171 | val 0.7222 | acc 0.700 tpr 0.500 tnr 0.900 | 26s
  ep  15 | train 0.3068 | val 0.8362 | acc 0.675 tpr 0.850 tnr 0.500 | 26s
  ep  20 | train 0.2481 | val 0.7294 | acc 0.700 tpr 0.650 tnr 0.750 | 25s
  ep  25 | train 0.1837 | val 0.8986 | acc 0.675 tpr 0.800 tnr 0.550 | 25s
  early stop 26, best 11
  >>> TEST Acc=67.5% TPR=45.0% TNR=90.0% | params=28,799,426 train=10.9min inf=14.65ms 212.74GFLOPs best_ep=11


In [14]:
results["mm"].append(run_seed(7, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_cnnpre_mm"))

F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  MedicalNet: 12 missing, 0 unexpected keys
  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_mm seed 7 ---
  ep   1 | train 0.7290 | val 0.6542 | acc 0.675 tpr 0.800 tnr 0.550 | 25s
  ep   5 | train 0.5029 | val 0.6878 | acc 0.625 tpr 0.900 tnr 0.350 | 25s
  ep  10 | train 0.3688 | val 0.6828 | acc 0.725 tpr 0.750 tnr 0.700 | 25s
  ep  15 | train 0.2450 | val 0.7749 | acc 0.675 tpr 0.600 tnr 0.750 | 25s
  ep  20 | train 0.2084 | val 0.8024 | acc 0.700 tpr 0.650 tnr 0.750 | 26s
  ep  25 | train 0.1932 | val 0.8075 | acc 0.750 tpr 0.750 tnr 0.750 | 26s
  early stop 25, best 3
  >>> TEST Acc=75.0% TPR=60.0% TNR=90.0% | params=28,799,426 train=10.4min inf=14.02ms 212.74GFLOPs best_ep=3


In [15]:
results["mm"].append(run_seed(123, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_cnnpre_mm"))

F:\mamba_model/MedicalNet\models\resnet.py:173: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
C:\Users\sammy\AppData\Local\Temp\ipykernel_7868\1694135348.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

  MedicalNet: 12 missing, 0 unexpected keys
  MedicalNet: 12 missing, 0 unexpected keys

--- v7_roi_cnnpre_mm seed 123 ---
  ep   1 | train 0.6578 | val 0.6846 | acc 0.625 tpr 0.300 tnr 0.950 | 26s
  ep   5 | train 0.5165 | val 0.6131 | acc 0.750 tpr 0.550 tnr 0.950 | 26s
  ep  10 | train 0.4530 | val 0.6298 | acc 0.725 tpr 0.700 tnr 0.750 | 25s
  ep  15 | train 0.3289 | val 0.7028 | acc 0.725 tpr 0.500 tnr 0.950 | 24s
  ep  20 | train 0.2826 | val 0.6731 | acc 0.700 tpr 0.600 tnr 0.800 | 24s
  ep  25 | train 0.1906 | val 0.7359 | acc 0.675 tpr 0.700 tnr 0.650 | 25s
  early stop 26, best 11
  >>> TEST Acc=70.0% TPR=55.0% TNR=85.0% | params=28,799,426 train=10.8min inf=13.73ms 212.74GFLOPs best_ep=11


In [16]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    fl = [r['flops'] for r in rs if r['flops']]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | {f'{np.mean(fl)/1e9:.2f}GFLOPs' if fl else 'N/A'} "
          f"| seeds={[r['seed'] for r in rs]}")

print(f"=== v7 ROI Vision Mamba — {TAG} augmentation, 200-subject cohort ===")
print(f"    seeds {INCLUDE_SEEDS}\n")
for k, n in [('mri','MRI-only  '), ('pet','PET-only  '), ('mm','Multimodal')]:
    summarize(results[k], n)

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS} (all seeds retained)")

=== v7 ROI Vision Mamba — tio_cnnpre augmentation, 200-subject cohort ===
    seeds [1, 7, 123]

MRI-only  : Acc=66.7±8.8% | TPR=68.3±24.7% | TNR=65.0±30.0% | Params=14,399,714 | Train=5.4m | Inf=7.80ms | 106.37GFLOPs | seeds=[1, 7, 123]
PET-only  : Acc=69.2±6.3% | TPR=60.0±8.7% | TNR=78.3±7.6% | Params=14,399,714 | Train=5.9m | Inf=7.85ms | 106.37GFLOPs | seeds=[1, 7, 123]
Multimodal: Acc=70.8±3.8% | TPR=53.3±7.6% | TNR=88.3±2.9% | Params=28,799,426 | Train=10.7m | Inf=14.13ms | 212.74GFLOPs | seeds=[1, 7, 123]

saved D:/mamba_model/v7_roi_tio_cnnpre_results.json (all seeds retained)


In [18]:
# GFLOPs for one forward pass, batch size 1 -- pretrained ResNet-10 tokenisation
#  Architecture alone determines these figures, so no training, checkpoints
#  or data are needed.
import copy
from torch.utils.flop_counter import FlopCounterMode
from mambapy.vim import VMamba as _VMamba


def count_flops(model, inputs):
    """Returns (conv_linear, scan, n_tokens) for one forward pass."""
    m = copy.deepcopy(model).eval().cpu()
    inputs = [t.detach().cpu() for t in inputs]
    mamba_log, handles = [], []

    def mk(mod):
        def hook(_, inp, __):
            c = mod.config
            mamba_log.append(dict(L=inp[0].shape[1], ed=c.d_inner, n=c.d_state,
                                  layers=c.n_layers,
                                  bi=getattr(c, "bidirectional", False)))
        return hook
    for mod in m.modules():
        if isinstance(mod, _VMamba):
            handles.append(mod.register_forward_hook(mk(mod)))

    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        m(*inputs)
    for h in handles:
        h.remove()

    cl = counter.get_total_flops()
    sc = sum(6 * r["ed"] * r["n"] * r["L"] * r["layers"] * (2 if r["bi"] else 1)
             for r in mamba_log)
    tokens = mamba_log[0]["L"] if mamba_log else 0
    del m
    return cl, sc, tokens


def report(name, model, inputs):
    p = sum(q.numel() for q in model.parameters() if q.requires_grad)
    cl, sc, tok = count_flops(model, inputs)
    print(f"  {name:28s} {p:>11,}p | {tok} tokens | "
          f"conv+linear {cl/1e9:.3f} + scan {sc/1e9:.4f} = {(cl+sc)/1e9:.3f} GFLOPs",
          flush=True)


roi = lambda: torch.randn(1, 6, 1, 64, 64, 64)
print("GFLOPs, one forward pass, batch size 1 -- pretrained ResNet-10 tokenisation\n")
report("ResNet-10 (unimodal)",   VisionMambaModel().cpu(),           [roi()])
report("ResNet-10 (multimodal)", MultimodalVisionMambaModel().cpu(), [roi(), roi()])

GFLOPs, one forward pass, batch size 1 -- pretrained ResNet-10 tokenisation

  ResNet-10 (unimodal)          14,399,714p | 6 tokens | conv+linear 106.175 + scan 0.0001 = 106.175 GFLOPs
  ResNet-10 (multimodal)        28,799,426p | 6 tokens | conv+linear 212.350 + scan 0.0003 = 212.351 GFLOPs
